# FastAPI Deep Dive: Lead Data Engineer Edition

**Target Audience**: Senior/Staff/Lead Data Engineers  
**Focus**: Dependency Injection, API Architecture, Real-World Patterns

---

## Why FastAPI for Data Engineering?

| Use Case | How FastAPI Helps |
|----------|-------------------|
| **Pipeline Triggers** | REST endpoints to kick off Spark/Airflow jobs |
| **Data Catalog APIs** | Serve metadata about tables, schemas |
| **Feature Serving** | Low-latency feature store access |
| **Internal Tools** | Admin dashboards, monitoring APIs |
| **Webhook Handlers** | Receive events from data platforms |

---

## 1. Dependency Injection: The Philosophy

### What Problem Does DI Solve?

```
WITHOUT DI (Tight Coupling):              WITH DI (Loose Coupling):
─────────────────────────                 ──────────────────────────
class PipelineService:                    class PipelineService:
    def __init__(self):                       def __init__(self, db, cache):
        self.db = PostgresDB()  # Hardcoded       self.db = db
        self.cache = Redis()    # Hardcoded       self.cache = cache
                                          
Problem:                                   Benefits:
- Can't test without real DB              - Easy to mock for tests
- Can't swap to MySQL                     - Swap Postgres → MySQL easily
- Hidden dependencies                     - Dependencies are explicit
```

### SOLID Principle: Dependency Inversion

> "High-level modules should not depend on low-level modules. Both should depend on abstractions."

In FastAPI, `Depends()` is how we inject those abstractions.

---

## 2. Dependency Injection Patterns

### Pattern 1: Function Dependencies (Simple)

In [1]:
from fastapi import FastAPI, Depends

app = FastAPI()

# Simple function that returns something
def get_settings():
    """
    Real-world: Load config from env vars or file.
    Called ONCE per request (FastAPI caches within request).
    """
    return {"env": "production", "debug": False}

@app.get("/config")
async def show_config(settings: dict = Depends(get_settings)):
    return settings

print("Pattern 1: Simple function returns value, injected via Depends()")

Pattern 1: Simple function returns value, injected via Depends()


### Pattern 2: Generator Dependencies (Lifecycle Management)

**Critical for Data Engineering**: Database connections, file handles, Spark sessions.

In [2]:
from contextlib import contextmanager

def get_db_session():
    """
    Generator dependency with cleanup.
    
    LIFECYCLE:
    1. Request comes in
    2. Code BEFORE yield runs (create session)
    3. Yield returns the session to endpoint
    4. Endpoint uses session
    5. Response sent to client
    6. Code AFTER yield runs (cleanup)
    """
    print("[DB] Opening connection...")
    session = "MockDBSession"  # Real: SessionLocal()
    try:
        yield session  # Endpoint gets this
    finally:
        print("[DB] Closing connection...")
        # Real: session.close()

@app.get("/pipelines")
async def list_pipelines(db = Depends(get_db_session)):
    # db is available here
    return {"db": db, "pipelines": []}
    # After return, finally block runs

print("Pattern 2: Generator with 'yield' for setup/teardown")

Pattern 2: Generator with 'yield' for setup/teardown


### Pattern 3: Class Dependencies (Stateful)

**When to use**: When you need to encapsulate related logic or use `__call__`.

In [3]:
class Pagination:
    """
    Class-based dependency.
    __init__ parameters become query params automatically!
    """
    def __init__(self, skip: int = 0, limit: int = 10):
        self.skip = skip
        self.limit = limit
        
    def apply(self, query):
        return query[self.skip:self.skip + self.limit]

@app.get("/jobs")
async def list_jobs(pagination: Pagination = Depends()):
    # With Depends(), FastAPI instantiates Pagination for you
    # URL: /jobs?skip=10&limit=5
    all_jobs = list(range(100))
    return pagination.apply(all_jobs)

print("Pattern 3: Class dependencies with automatic param parsing")

Pattern 3: Class dependencies with automatic param parsing


### Pattern 4: Nested Dependencies (Dependency Chains)

**Real-World Example**: Auth → User → Permissions

```
Request
   ↓
get_token()  ←────────── Depends
   ↓
get_current_user(token)  ←── Depends(get_token)
   ↓
check_permissions(user)  ←── Depends(get_current_user)
   ↓
Endpoint
```

In [4]:
from fastapi import Header, HTTPException

def get_token(authorization: str = Header(...)):
    """Level 1: Extract token from header."""
    if not authorization.startswith("Bearer "):
        raise HTTPException(401, "Invalid auth header")
    return authorization.split(" ")[1]

def get_current_user(token: str = Depends(get_token)):
    """Level 2: Validate token, return user (depends on get_token)."""
    # Real: decode JWT, lookup user
    if token == "admin-token":
        return {"id": 1, "role": "admin"}
    raise HTTPException(401, "Invalid token")

def require_admin(user: dict = Depends(get_current_user)):
    """Level 3: Check permissions (depends on get_current_user)."""
    if user["role"] != "admin":
        raise HTTPException(403, "Admin required")
    return user

@app.delete("/pipelines/{id}")
async def delete_pipeline(id: int, admin = Depends(require_admin)):
    # Only admins can delete
    return {"deleted": id, "by": admin}

print("Pattern 4: Nested dependencies form a chain")

Pattern 4: Nested dependencies form a chain


---

## 3. Testing with Dependency Overrides

**The Power of DI**: Swap real dependencies for mocks during tests.

In [5]:
from fastapi.testclient import TestClient

# In tests, override the real DB with a mock
def override_get_db():
    return "MockTestDB"

# Apply override
app.dependency_overrides[get_db_session] = override_get_db

# Now all endpoints using get_db_session get MockTestDB
client = TestClient(app)
response = client.get("/pipelines")
print(f"Response with mocked DB: {response.json()}")

# Cleanup
app.dependency_overrides.clear()

print("\nTesting: Use dependency_overrides to swap real deps for mocks")

Response with mocked DB: {'db': 'MockTestDB', 'pipelines': []}

Testing: Use dependency_overrides to swap real deps for mocks


---

## 4. Data Engineering API Patterns

### Pattern: ETL Pipeline Trigger API

In [6]:
from pydantic import BaseModel
from typing import Optional
from datetime import datetime

class PipelineRun(BaseModel):
    pipeline_id: str
    trigger_date: Optional[datetime] = None
    parameters: Optional[dict] = None

class PipelineResponse(BaseModel):
    run_id: str
    status: str
    message: str

@app.post("/pipelines/{pipeline_id}/trigger", response_model=PipelineResponse)
async def trigger_pipeline(
    pipeline_id: str,
    run_config: PipelineRun,
    db = Depends(get_db_session)
):
    """
    Trigger an ETL pipeline.
    
    Real-world: This would call Airflow/Databricks API.
    """
    run_id = f"run_{pipeline_id}_{datetime.now().strftime('%Y%m%d%H%M%S')}"
    
    # Real: Submit to Airflow
    # airflow_client.trigger_dag(pipeline_id, run_config.parameters)
    
    return PipelineResponse(
        run_id=run_id,
        status="triggered",
        message=f"Pipeline {pipeline_id} queued for execution"
    )

print("Data Engineering: Pipeline trigger endpoint")

Data Engineering: Pipeline trigger endpoint


### Pattern: Data Quality Check API

In [7]:
class DataQualityCheck(BaseModel):
    table_name: str
    check_type: str  # "null_check", "uniqueness", "freshness"
    column: Optional[str] = None

class QualityResult(BaseModel):
    passed: bool
    details: dict
    timestamp: datetime

@app.post("/quality/check", response_model=QualityResult)
async def run_quality_check(check: DataQualityCheck):
    """
    Run a data quality check.
    
    Real-world: Query data warehouse, run Great Expectations.
    """
    # Simulated check
    result = {
        "table": check.table_name,
        "check": check.check_type,
        "rows_checked": 1000000,
        "failures": 0
    }
    
    return QualityResult(
        passed=True,
        details=result,
        timestamp=datetime.now()
    )

print("Data Engineering: Quality check endpoint")

Data Engineering: Quality check endpoint


---

## 5. Lead Engineer Interview Questions

### Q1: When would you NOT use Dependency Injection?

**Answer**:
- **Simple utilities** that have no side effects (pure functions)
- **Constants** that never change
- When injecting adds complexity without benefit

### Q2: Explain the difference between `Depends()` with and without parentheses.

```python
# Without parentheses: Dependency is called for you
Depends(get_db)  # FastAPI calls get_db()

# With parentheses on CLASS: FastAPI instantiates class
Depends(Pagination)  # FastAPI calls Pagination(skip=..., limit=...)
# OR
Depends()  # When type hint provides the class: pagination: Pagination = Depends()
```

### Q3: How do you share a single DB connection across an entire request?

**Answer**: FastAPI caches dependency results within a request. If `get_db` is called multiple times in nested dependencies, only ONE connection is created.

### Q4: How would you implement rate limiting in FastAPI?

**Answer**: Use a dependency that checks Redis/cache for request counts:
```python
async def rate_limit(request: Request, redis = Depends(get_redis)):
    key = f"rate:{request.client.host}"
    count = await redis.incr(key)
    if count == 1:
        await redis.expire(key, 60)  # 60 second window
    if count > 100:
        raise HTTPException(429, "Rate limit exceeded")
```

### Q5: What's the difference between `BackgroundTasks` and Celery?

| Feature | BackgroundTasks | Celery |
|---------|-----------------|--------|
| **Scope** | Same process | Separate workers |
| **Persistence** | None (memory) | Redis/RabbitMQ |
| **Failure Handling** | None | Retries, dead-letter |
| **Use Case** | Light tasks (email) | Heavy tasks (ETL) |

---

## 6. Architecture Patterns Summary

### Request Flow
```
Client → Uvicorn → Middleware → Router → Dependencies → Endpoint → Response Model → Client
```

### Dependency Resolution Order
```
1. Path parameters extracted
2. Query parameters extracted
3. Dependencies resolved (depth-first, cached per request)
4. Request body parsed
5. Endpoint function called
6. Response model applied
7. Dependency cleanup (finally blocks)
```

### Production Checklist
- [ ] Use `async def` for I/O-bound endpoints
- [ ] Implement proper error handling with custom exceptions
- [ ] Add request ID middleware for tracing
- [ ] Configure connection pooling for DB
- [ ] Set up health check endpoint
- [ ] Use Pydantic for ALL request/response models